## Project & File Organization

*This is the boring-but-important part: where files go, how notebooks find them, and how you save your work so others (and future you) can understand it.*

**By the end you will be able to:**

1. Lay out a project so data, code, and results each have an obvious home.
2. Point every notebook at the project with **one auto-detected line** — no more
   `/Users/yourname/Desktop/...` paths that break on the next machine.
3. Save **intermediate data** in a format that doesn't alter your data types.
4. Save and store **figures, tables, and fitted models** so results are reproducible and shareable.

## Why this matters

- **The hardcoded path.** `pd.read_csv("/Users/ME/Desktop/data.csv")` runs on YOUR laptop
  and nowhere else. A collaborator has to hunt down paths.
- **The mystery file.** `results_final_v3_USE_THIS.csv` — made by which script? From which raw
  data? what does `results_final_v4_ACTUALLY_USE.csv` mean?
- **The edited raw file.** Someone "fixed" a value directly in the spreadsheet. The original
  measurement is now altered and we can't reproduce results relying on the orginal raw data.
- **The 6-hour model you didn't save.** You fit it, looked at the accuracy, closed the notebook.
  It's gone. You get to fit it again.

The fix for all four is the same: **a predictable structure, relative paths, an immutable raw
copy, and the habit of saving your outputs.**

## Cookie-cutter layout

This is an adapted version of *Cookiecutter Data Science* conventions. I don't follow it exactly and treat some things a bit differetly. BUT, the single most important rule: **`data/raw/` is read-only.** Data flows one direction —
`raw → interim → processed` — and every transformation is done in code so everyone can see what was done and repeat it if needed. If you never edit the raw data, you can always rebuild everything else from scratch.

**This is the general layout I like to use:**

```
tem1-fitness/
├── .projectroot          <- empty marker file. can be used to automatically detect BASE_DIR
├── README.md             <- project info
├── requirements.txt      <- libraries and versioning
├── paths.py              <- ONE place to define our paths
├── data/
│   ├── raw/              <- original data. NEVER edit by hand or overwrite.
│   ├── interim/          <- cleaned / partially transformed checkpoints
│   └── processed/        <- final, model-ready tables
├── notebooks/            <- the analysis
├── src/                  <- reusable code (helpers and such)
├── models/               <- fitted models & weights (+ their metadata)
└── results/
    ├── figures/          <- plots
    └── tables/           <- summary tables
```

### Some setup

In [1]:
from pathlib import Path
import json
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
print ('done')

## Build the scaffold (only need to run once)

The function below *creates* the whole structure, drops in a README, a `requirements.txt`, the
`.projectroot` marker, and the shared `paths.py`. It's **idempotent** — running it again won't
overwrite anything because we set: `exist_ok=True`.

**You set an explicit `PROJECT_DIR`**
(e.g. `Path.home() / "projects" / "tem1-fitness"`, or `Path("/Users/me/Desktop/Projects/tem1-fitness")`) whenever you want — you're deciding where the project lives. Once it exists, the
`.projectroot` marker makes every downstream notebook find it.

**Using git?** If not, leave `USE_GIT = False` no `.gitignore`
and no `.gitkeep`

In a real project this notebook would live at `notebooks/Part00_Setup.ipynb`. For this demo its sitting right here.

In [2]:
#this bit will create a paths.py file in your directory. It will assign paths relative to the BASE_DIR you set in this script
#you can modify the paths in that paths.py file.
from pathlib import Path
import argparse


# ---------------------------------------------------------------------------
# The paths.py that gets dropped into every new project
# ---------------------------------------------------------------------------
PATHS_PY = '''"""
paths.py -- one source of truth for every path in the project.

Two ways to use it:
  1. simplest: paste the body of this file into a cell at the top of each notebook.
  2. keep this file and `from paths import *` -- but Python must be able to SEE it on
     the import path. That means the notebook lives inside the project; from notebooks/
     put the project root on sys.path first:
         import sys; from pathlib import Path
         root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
         sys.path.insert(0, str(root))
         from paths import *
Either way, no notebook contains an absolute path except (at most) the project root below.
"""
from pathlib import Path
import os
import matplotlib.pyplot as plt

def find_project_root(explicit=None, marker=".projectroot"):
    """Locate the project root, in priority order:
       1. an explicit path you pass in,
       2. the $PROJECT_ROOT environment variable,
       3. walking UP from the working directory to find the marker file.
    If none are found we RAISE instead of silently guessing the cwd -- a wrong guess
    would scatter data/ and models/ into the wrong folder."""
    if explicit is not None:
        return Path(explicit).expanduser().resolve()
    if os.environ.get("PROJECT_ROOT"):
        return Path(os.environ["PROJECT_ROOT"]).expanduser().resolve()
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / marker).exists():
            return folder
    raise FileNotFoundError(
        f"No project root found (looked for '{marker}' from {here} upward). "
        f"Fix with one of: run your project's setup; move this notebook into the "
        f"project's notebooks/ folder; pass find_project_root('/path/to/project'); or set $PROJECT_ROOT.")


# Leave PROJECT_ROOT = None to auto-detect (the notebook must live inside the project).
# Set an explicit path only when running from OUTSIDE the tree.
PROJECT_ROOT = None
BASE_DIR = find_project_root(PROJECT_ROOT)
DATA      = BASE_DIR / "data"
RAW       = DATA / "raw"                  # read-only, never edited by hand
INTERIM   = DATA / "interim"              # cleaned / partially processed
PROCESSED = DATA / "processed"            # model-ready
MODELS    = BASE_DIR / "models"           # top-level: models are first-class artifacts
RESULTS   = BASE_DIR / "results"
FIGURES   = RESULTS / "figures"
TABLES    = RESULTS / "tables"

for _d in (RAW, INTERIM, PROCESSED, FIGURES, MODELS, TABLES):
    _d.mkdir(parents=True, exist_ok=True)
'''
README = """# {name}

Project layout:
  data/raw/        immutable source data (never edit by hand)
  data/interim/    cleaned / intermediate checkpoints
  data/processed/  model-ready tables
  notebooks/       analysis notebooks, in order
  src/             reusable code / helpers
  models/          fitted models + their metadata
  results/         figures & tables

Everything except data/raw/ is rebuilt by code, so the raw folder stays untouched.
"""

REQUIREMENTS = "pandas\nnumpy\nmatplotlib\nseaborn\nscikit-learn\njoblib\npyarrow\n"

GITIGNORE = (
    "# keep code + structure in git; keep bulky/derived data out\n"
    "data/raw/*\ndata/interim/*\ndata/processed/*\n!**/.gitkeep\n"
    "__pycache__/\n.ipynb_checkpoints/\n*.pyc\n"
    "# large model weights go in git-LFS, not committed directly\n"
)
# using git/GitHub?
USE_GIT = False   # set True to write .gitignore + .gitkeep placeholders
# --------------------------------------------


def scaffold(root: Path, use_git=USE_GIT):
    root = Path(root)
    dirs = ["data/raw", "data/interim", "data/processed",
            "notebooks", "src", "models", "results/figures", "results/tables"]
    for d in dirs:
        (root / d).mkdir(parents=True, exist_ok=True)
        if use_git:
            (root / d / ".gitkeep").touch()  # git needs a file to track an empty dir

    (root / ".projectroot").touch()   # path marker
    (root / "paths.py").write_text(PATHS_PY)
    (root / "README.md").write_text(#update contents of the README
        "# TEM-1 fitness project\n\n"
        "Run notebooks in order: Part00 (this) -> Part0 (EDA) -> Part1 -> Part2.\n"
        "Raw data in data/raw/ is immutable; everything else is rebuilt by code.\n")
    (root / "requirements.txt").write_text(#Some standard packages to start
        "pandas\nnumpy\nmatplotlib\nseaborn\nscikit-learn\njoblib\npyarrow\n")
    if use_git:
        (root / ".gitignore").write_text(
            "# keep code + structure in git; keep bulky/derived data out\n"
            "data/raw/*\ndata/interim/*\ndata/processed/*\n!**/.gitkeep\n"
            "__pycache__/\n.ipynb_checkpoints/\n*.pyc\n"
            "# large model weights go in git-LFS, not committed directly\n")
    return root
print ('done')

In [7]:
# WHERE should the project live? Make this explicit
#PROJECT_DIR = Path.home() / "projects" / "tem1-fitness"
PROJECT_DIR = Path("/Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2") #im setting an explicit path here
# ---------------------------------------------------------------------------------

print("current working directory :", Path.cwd())
print("project will be created at:", PROJECT_DIR.resolve())

PROJECT = scaffold(PROJECT_DIR)
print("scaffolded ->", PROJECT.resolve(), "| git files:", USE_GIT)

current working directory : /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2
project will be created at: /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2
scaffolded -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2 | git files: False


## The paths
Once you run the scaffold script, you should see a project setup like below.

In [8]:
BASE_DIR = PROJECT_DIR
DATA      = BASE_DIR / "data"
RAW       = DATA / "raw"
INTERIM   = DATA / "interim"
PROCESSED = DATA / "processed"
MODELS    = BASE_DIR / "models" 
RESULTS   = BASE_DIR / "results"
FIGURES   = RESULTS / "figures"
TABLES    = RESULTS / "tables"

## Put the raw data where it belongs

We copy all of the original/raw data into `data/raw/`. From here on, notebooks read from `RAW` and treat it
as read-only.

## Intermediate data — checkpoint your work

We **dont** want to spend time re-cleaning, filtering and formatting our data (fixing categories, dropping duplicates, removing impossible values) inside every downstream notebook. Save a checkpoint to `data/interim/` and re-load it for downstream tasks.

But *how* you save matters. CSVs are great because they are full interoperable, you can pass them to R, to Prism, whatever. However, CSVs do NOT save data types. CSV is plain text: it has no idea what a category, an ordering, a boolean, or a datetime is, so it flattens everything to strings.

| format | preserves dtypes? | portable? | use it for |
|---|---|---|---|
| **CSV** | no | everywhere | raw data, exports, widely recognized |
| **Parquet** | **yes** | Python/R/Spark/etc. | interim & processed checkpoints |
| **pickle / joblib** | yes | Python only, version-fragile | quick local caches, fitted models |

You dont have to use Parquet, but it is nice for these intermediate/processed data. It's compressed, fast, and
keeps your data types.

I use CSV all the time, especially if the data is not too large and it will be shared. I wouldn't share a parquet file with someone that doesn't work in R/Python.

Just note you will have to make sure the data types are what you want when reading in. This is good practice to verify even if you use parquet files, honestly.

**Good for large projects with LOTS of data** Save a recording of *how* the file was made — what it came from, when, and with what code — so future-you never has to guess. *I dont do this for smaller projects.*

## Saving results — figures

Regenerating every plot when needed can be annoying. Save them with **descriptive names** into
`results/figures/`, at a resolution good enough for slides. The `save_fig` helper we defined
saves `dpi` / `bbox_inches` so every figure comes out consistent.

In [9]:
#I would dump this at the top of the script so its available--can adjust the resolution, format, etc.

def save_fig(name, fig=None, dpi=150):
    """Save figure into results/figures"""
    fig = fig or plt.gcf()
    path = FIGURES / f"{name}.pdf" #i like PDF, PNG is good too
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path

## Saving results — tables

The summary tables that go into a report are results too, and different from the `data`. Save them to `results/tables/`.

## Saving results — fitted models & weights

You want to save your trained models. Losing it means re-running the fit. Saving it also means we can share it and others can reproduce your predictions without your data.

1. **Save model weights with `joblib`** 
2. **Never save a model without its metadata.** A lone `model.joblib` isnt enough. Store a
   JSON file next to it: what features, what data, what score, what library versions. A pickle
   is only loadable by compatible versions, so be sure to track them.

In [10]:
def tree(root: Path, prefix=""):
    entries = sorted([p for p in root.iterdir() if p.name != ".git"],
                     key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            tree(p, prefix + ("    " if last else "│   "))

print(PROJECT.name + "/")
tree(PROJECT)

Beta Lactam ML v2/
├── .ipynb_checkpoints/
│   └── project_setup-checkpoint.ipynb
├── data/
│   ├── interim/
│   ├── processed/
│   └── raw/
├── models/
├── notebooks/
├── results/
│   ├── figures/
│   └── tables/
├── src/
├── .projectroot
├── README.md
├── paths.py
├── project_setup.ipynb
└── requirements.txt


With this we just have to set up our directory once and then just refer to
`RAW`, `INTERIM`, `FIGURES`, `MODELS`, etc - not an absolute path.

The structure is boring on purpose. Boring is what reproducible looks like — and it means the
next person to open your project (very often future you) can run it start to finish without a
single "wait, where's the data?"